# **Day 53: Project - Satellite Image Catalogue**

In this project, we use all the OOP ideas from Day 41 to Day 52 together.

## **The Story**
A Remote Sensing lab keeps many satellite images of West Bengal on its computers. Students ask questions like:
- "Give me a clear Sentinel-2 image of Kolkata from winter 2024."
- "Which images have less than 10% cloud?"
- "How much data do we have for each satellite?"

We will build a small **catalogue system** to answer these questions. It will also let students **borrow** (reserve) an image for their project, just like a library.

**Features:**
1. Different satellite types (using **inheritance** and **abstraction**)
2. Scene information stored with **dataclasses**
3. Processing levels and seasons as **enums**
4. Safe borrowing with **encapsulation** and **custom exceptions**
5. Search with many filters

(All scene data is made up.)

In [1]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import date
from enum import Enum

class CatalogueError(Exception):
    pass

class Level(Enum):
    L1 = "Level-1 (raw, top of atmosphere)"
    L2 = "Level-2 (surface reflectance, ready to use)"

class Season(Enum):
    WINTER = (12, 1, 2)
    SUMMER = (3, 4, 5)
    MONSOON = (6, 7, 8, 9)
    POST_MONSOON = (10, 11)

    @classmethod
    def of(cls, d):
        for s in cls:
            if d.month in s.value:
                return s

## **Part 1: Sensors (Inheritance and Abstraction)**

In [2]:
class Sensor(ABC):
    @property
    @abstractmethod
    def name(self): ...

    @property
    @abstractmethod
    def resolution_m(self): ...

    @property
    @abstractmethod
    def size_mb_per_scene(self): ...

    def __repr__(self):
        return f"{self.name} ({self.resolution_m} m)"

class Landsat8(Sensor):
    name = "Landsat 8"
    resolution_m = 30
    size_mb_per_scene = 950

class Sentinel2(Sensor):
    name = "Sentinel-2"
    resolution_m = 10
    size_mb_per_scene = 800

class Resourcesat(Sensor):           # An Indian (ISRO) satellite
    name = "Resourcesat-2A"
    resolution_m = 23.5
    size_mb_per_scene = 600

print(Landsat8(), Sentinel2(), Resourcesat())

Landsat 8 (30 m) Sentinel-2 (10 m) Resourcesat-2A (23.5 m)


- Here, the child classes fill the abstract properties with simple **class variables**. This also satisfies the abstract class rules.

## **Part 2: The Scene (Dataclass)**

In [3]:
@dataclass
class Scene:
    scene_id: str
    sensor: Sensor
    acquired: date
    cloud_cover: float
    area: str
    level: Level = Level.L2
    borrowed_by: str | None = field(default=None, repr=False)

    @property
    def season(self):
        return Season.of(self.acquired)

    @property
    def available(self):
        return self.borrowed_by is None

    def __str__(self):
        status = "available" if self.available else f"with {self.borrowed_by}"
        return (f"{self.scene_id:<24}{self.sensor.name:<15}{self.acquired}  "
                f"cloud {self.cloud_cover:>4.1f}%  {self.area:<11}{status}")

## **Part 3: The Catalogue (Encapsulation)**
The list of scenes is kept **private** (`_scenes`). Students can only use the catalogue through its methods, so nobody can break the data by mistake.

In [4]:
class Catalogue:
    MAX_BORROW = 2

    def __init__(self, lab_name):
        self.lab_name = lab_name
        self._scenes = {}
        self._borrowed = {}           # student -> list of scene ids

    def add(self, scene):
        if scene.scene_id in self._scenes:
            raise CatalogueError(f"{scene.scene_id} already exists")
        self._scenes[scene.scene_id] = scene

    def search(self, sensor=None, area=None, season=None, max_cloud=100, year=None, only_available=False):
        results = []
        for s in self._scenes.values():
            if sensor and s.sensor.name != sensor:
                continue
            if area and s.area != area:
                continue
            if season and s.season != season:
                continue
            if year and s.acquired.year != year:
                continue
            if s.cloud_cover > max_cloud:
                continue
            if only_available and not s.available:
                continue
            results.append(s)
        return sorted(results, key=lambda s: s.cloud_cover)

    def borrow(self, student, scene_id):
        scene = self._scenes.get(scene_id)
        if scene is None:
            raise CatalogueError(f"No scene called {scene_id}")
        if not scene.available:
            raise CatalogueError(f"{scene_id} is already with {scene.borrowed_by}")
        if len(self._borrowed.get(student, [])) >= self.MAX_BORROW:
            raise CatalogueError(f"{student} already has {self.MAX_BORROW} scenes")
        scene.borrowed_by = student
        self._borrowed.setdefault(student, []).append(scene_id)

    def give_back(self, student, scene_id):
        self._borrowed[student].remove(scene_id)
        self._scenes[scene_id].borrowed_by = None

    def storage_report(self):
        report = {}
        for s in self._scenes.values():
            report[s.sensor.name] = report.get(s.sensor.name, 0) + s.sensor.size_mb_per_scene
        return report

    def __len__(self):
        return len(self._scenes)

## **Part 4: Using the Catalogue**

In [5]:
L8, S2, RS = Landsat8(), Sentinel2(), Resourcesat()
cat = Catalogue("RS & GIS Lab")

data = [
    ("S2A_45QXF_20240105", S2, date(2024, 1, 5), 3.1, "Kolkata"),
    ("S2B_45QXF_20240120", S2, date(2024, 1, 20), 0.4, "Kolkata"),
    ("S2A_45QXF_20240705", S2, date(2024, 7, 5), 81.0, "Kolkata"),
    ("LC08_138044_20231110", L8, date(2023, 11, 10), 2.3, "Kolkata"),
    ("LC08_139045_20240214", L8, date(2024, 2, 14), 7.8, "Midnapore"),
    ("S2B_45RXL_20241205", S2, date(2024, 12, 5), 5.5, "Darjeeling"),
    ("S2A_45RXL_20240810", S2, date(2024, 8, 10), 64.0, "Darjeeling"),
    ("R2A_107055_20240118", RS, date(2024, 1, 18), 1.2, "Sundarbans"),
    ("LC08_138045_20240125", L8, date(2024, 1, 25), 0.9, "Sundarbans"),
]
for sid, sensor, d, cloud, area in data:
    cat.add(Scene(sid, sensor, d, cloud, area))

print(f"{cat.lab_name}: {len(cat)} scenes in the catalogue")

RS & GIS Lab: 9 scenes in the catalogue


In [6]:
print("Clear Sentinel-2 images of Kolkata in winter 2024:")
for s in cat.search(sensor="Sentinel-2", area="Kolkata", season=Season.WINTER, max_cloud=10, year=2024):
    print("  ", s)

Clear Sentinel-2 images of Kolkata in winter 2024:
   S2B_45QXF_20240120      Sentinel-2     2024-01-20  cloud  0.4%  Kolkata    available
   S2A_45QXF_20240105      Sentinel-2     2024-01-05  cloud  3.1%  Kolkata    available


In [7]:
print("All scenes with less than 5% cloud:")
for s in cat.search(max_cloud=5):
    print("  ", s)

All scenes with less than 5% cloud:
   S2B_45QXF_20240120      Sentinel-2     2024-01-20  cloud  0.4%  Kolkata    available
   LC08_138045_20240125    Landsat 8      2024-01-25  cloud  0.9%  Sundarbans available
   R2A_107055_20240118     Resourcesat-2A 2024-01-18  cloud  1.2%  Sundarbans available
   LC08_138044_20231110    Landsat 8      2023-11-10  cloud  2.3%  Kolkata    available
   S2A_45QXF_20240105      Sentinel-2     2024-01-05  cloud  3.1%  Kolkata    available


In [8]:
# Borrowing scenes
actions = [
    ("Rina", "S2B_45QXF_20240120"),
    ("Arko", "S2B_45QXF_20240120"),       # Already borrowed by Rina
    ("Rina", "LC08_138045_20240125"),
    ("Rina", "R2A_107055_20240118"),      # Rina already has 2
    ("Tuhin", "S2A_99XXX_20240101"),      # Does not exist
]
for student, sid in actions:
    try:
        cat.borrow(student, sid)
        print(f"OK   : {student} borrowed {sid}")
    except CatalogueError as e:
        print(f"ERROR: {e}")

OK   : Rina borrowed S2B_45QXF_20240120
ERROR: S2B_45QXF_20240120 is already with Rina
OK   : Rina borrowed LC08_138045_20240125
ERROR: Rina already has 2 scenes
ERROR: No scene called S2A_99XXX_20240101


In [9]:
print("Available Sundarbans scenes now:")
for s in cat.search(area="Sundarbans", only_available=True):
    print("  ", s)

cat.give_back("Rina", "LC08_138045_20240125")
print("\nAfter Rina returns one scene:")
for s in cat.search(area="Sundarbans", only_available=True):
    print("  ", s)

Available Sundarbans scenes now:
   R2A_107055_20240118     Resourcesat-2A 2024-01-18  cloud  1.2%  Sundarbans available

After Rina returns one scene:
   LC08_138045_20240125    Landsat 8      2024-01-25  cloud  0.9%  Sundarbans available
   R2A_107055_20240118     Resourcesat-2A 2024-01-18  cloud  1.2%  Sundarbans available


In [10]:
print("Storage used by each satellite:")
for sensor, mb in cat.storage_report().items():
    print(f"  {sensor:<15}{mb / 1024:6.2f} GB")

Storage used by each satellite:
  Sentinel-2       3.91 GB
  Landsat 8        2.78 GB
  Resourcesat-2A   0.59 GB


## **Which OOP Ideas Did We Use?**
| Idea | Where |
|---|---|
| **Abstraction** | `Sensor` abstract class |
| **Inheritance** | `Landsat8`, `Sentinel2`, `Resourcesat` |
| **Polymorphism** | Every sensor has `name`, `resolution_m`, `size_mb_per_scene` |
| **Encapsulation** | Private `_scenes` and `_borrowed` in `Catalogue` |
| **Dataclass** | `Scene` |
| **Enum** | `Level`, `Season` |
| **Custom exception** | `CatalogueError` |
| **Magic method** | `__len__`, `__str__`, `__repr__` |